In [79]:
import pdfplumber
import os 
import re
import importlib
import tiktoken
print("tiktoken version", importlib.metadata.version("tiktoken"))
from torch.utils.data import Dataset, DataLoader


tiktoken version 0.12.0


In [26]:
pdf_path="""F:/the-art-of-seduction-robert-greene.pdf"""
txt_path="F:/llm/art_of_seduction.txt"
txt_data=""

with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        text=page.extract_text()
        if text:
            txt_data += text + "\n"

with open(txt_path,"w",encoding="utf-8") as f:
    f.write(txt_data)


print("conversion completed")

conversion completed


In [16]:
with open("F:/llm/art_of_seduction.txt", "r",encoding="utf-8") as text:
    raw_text=text.read()

print("total number of characters", len(raw_text))


total number of characters 1397623


In [17]:


split_text = re.split(r"""[,.:;'!"()?]|--|(\s)""", raw_text)
print(split_text[:15])

split_text = [i.strip() for i in split_text if i and i.strip()]
print(split_text[:15])

tokens=sorted(set(split_text))
print(tokens[:10])

tokens.extend(["'","|unk|","|end_of_text|"])


['PENGUIN', ' ', 'BOOKS', '\n', 'THE', ' ', 'ART', ' ', 'OF', ' ', 'SEDUCTION', '\n', 'Robert', ' ', 'Greene']
['PENGUIN', 'BOOKS', 'THE', 'ART', 'OF', 'SEDUCTION', 'Robert', 'Greene', 'author', 'of', 'The', '48', 'Laws', 'of', 'Power']
['$31', '$37', '&', '*', '-', '-gods', '/', '/All', '/Seductive', '0']


In [18]:
vocab={token:id for id,token in enumerate(tokens)}




In [19]:
encoded=[vocab[item] for item in split_text  ]

In [20]:
id_to_text= {id:token for token,id in vocab.items()}
decoded= [id_to_text[id] for id in encoded]


In [21]:
class SimpleTokenizer():
    def __init__(self,vocab):
        self.str_to_id = vocab
        self.id_to_str ={id:token for token,id in vocab.items()}

    def encode(self,text):
        preprocessed= re.split(r'([,.:;?_!"()\']|--|\s)',text)
        preprocessed=[item.strip() for item in preprocessed if item.strip()]
        preprocessed=[item if item in self.str_to_id else "|unk|" for item in preprocessed ]
        ids=[self.str_to_id[item] for item in preprocessed ]
        return ids

    def decode(self,ids):
        text= " ".join([self.id_to_str[id] for id in ids])
        text=re.sub(r'\s+([,.:;?!"()\'])',r'\1',text)
        return text
         
        
        
        

In [32]:
tokenizer= SimpleTokenizer(vocab)

text = """"It's the last he painted, you know," 
           Mrs. Gisburn said with pardonable pride."""
id = tokenizer.encode(text)
print(id)

[21602, 3055, 21601, 17446, 19511, 13273, 11859, 15212, 21602, 21411, 13189, 21602, 21602, 21602, 21602, 21602, 17481, 21196, 21602, 16117, 21602]


In [33]:
tokenizer.decode(id)

"|unk| It' s the last he painted |unk| you know |unk| |unk| |unk| |unk| |unk| said with |unk| pride |unk|"

In [66]:
tokenizer= tiktoken.get_encoding("gpt2")

In [67]:
id=tokenizer.encode(""""It's the last he painted, you know," Mrs. Gisburn said with pardonable pride.""")

In [68]:
tokenizer.decode(id)

'"It\'s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

In [69]:
with open("F:/llm/art_of_seduction.txt", "r",encoding="utf-8") as text:
    raw_text=text.read()

In [70]:
enc_text=tokenizer.encode(raw_text)

In [71]:
dec_text=tokenizer.decode(enc_text)

In [72]:
context_size=4

x=enc_text[:context_size]
y=enc_text[1:context_size + 1]

In [73]:
for i in range(1,context_size + 1):
    context=enc_text[:i]
    desired=enc_text[i]

    print (context, "------>", desired)



[47] ------> 26808
[47, 26808] ------> 52
[47, 26808, 52] ------> 1268
[47, 26808, 52, 1268] ------> 39633


In [75]:
for i in range(1,context_size + 1):
    context=enc_text[:i]
    desired=enc_text[i]

    print (tokenizer.decode(context), "------>",tokenizer.decode([desired]))



P ------> ENG
PENG ------> U
PENGU ------> IN
PENGUIN ------>  BOOK


In [91]:
class GPTdataset(Dataset):
    def __init__(self,text,tokenizer,max_length,stride):
        self.input_id=[]
        self.output_id=[]

        token_ids=[tokenizer.encode(text,allowed_special={"<|endoftext|>"})]

        for i in range(0,len(token_ids)-maxlength,stride):

            input_chunck=token_ids[i:i +  max_length]
            output_chunck=token_ids[i+1:i+max_length +1]
            self.input_id.append(torch.tensor(input_chunck))
            self.output_id.append(torch.tensor(output_chunck))
            
    def __len__(self):
        return len(self.input_id)

    def __getitem__(self,idx):
        return self.input_id[idx], self.output_id[idx]